# **Setup**

In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

In [ ]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [ ]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [ ]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

# **Load Data**

In [ ]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [ ]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [ ]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_CF")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "topK": optuna_trial.suggest_int("topK", 50, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "similarity": "cosine",
        "normalize": True
    }
    
    validation_scores = []
    for URM_train, URM_validation in folds:        
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {len(validation_scores)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(np.mean(validation_scores), len(validation_scores))

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            raise optuna.TrialPruned()
        
        # Log fold performance
        optimizer.log_fold_performance(len(validation_scores), score)

    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [ ]:
bp = optimizer.get_best_params(STUDY_NAME)

def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "topK": optuna_trial.suggest_int("topK", max(10, bp["topK"]-50), bp["topK"]+50),
        "shrink": optuna_trial.suggest_int("shrink", max(0, bp["shrink"]-50), bp["shrink"]+50),
        "similarity": "cosine",
        "normalize": True
    }
    
    validation_scores = []
    for URM_train, URM_validation in folds:        
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {len(validation_scores)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(np.mean(validation_scores), len(validation_scores))

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            raise optuna.TrialPruned()
        
        # Log fold performance
        optimizer.log_fold_performance(len(validation_scores), score)

    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE

# **Train Model with best hyperparameter**

In [ ]:
# Train final model on train + validation with best hyperparameters
URM_train, URM_validation = folds[0]

bp = optimizer.get_best_params(STUDY_NAME+"_refined")

recommender = ItemKNNCFRecommender(URM_train + URM_validation)
recommender.fit(
    topK=bp["topK"],
    shrink=bp["shrink"],
    similarity="cosine",
    normalize=True
)
# Save the trained model
recommender.save_model(paths.MODEL_DIR)

In [ ]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, STUDY_NAME + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")